In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL
import database 
import gameSearch 
import datetime

from database import is_subscribed 
from database import subscriptions 
from database import sub_type 
from database import rental_lim

from datetime import datetime, timedelta

from gameSearch import search_Vid, search_Board


# Allows user to rent game (if avaliable) and adds it to Rental.txt

def game_Rent(user_ID, game_ID):
    
    
    if is_subscribed(user_ID):

        if search_Vid(game_ID) or search_Board(game_ID):

            rent_db = open('Rental.txt', 'r')
            rental_Date = datetime.now().strftime("%Y-%m-%d")

            rent_lim = rental_lim(user_ID)
            year = int(rental_Date.replace('-', ' ').split()[0])
            month = int(rental_Date.replace('-', ' ').split()[1])
            day = int(rental_Date.replace('-', ' ').split()[2])

            latest_return_Date = datetime(year, month, day) + timedelta(weeks=rent_lim)
            latest_return_Date = str(latest_return_Date).replace(" 00:00:00", "")

            for line in rent_db.readlines()[1:]:
                id = line.split(',')[0]
                returned = line.split(',')[2]
                if game_ID == id and returned == 'N/A':
                    return "Failed to rent! Game is already rented!"
                    return
        
            else: 
                rent_db.close()
                write_db = open('Rental.txt', 'a')
                entry = f"{game_ID},{rental_Date},N/A,{user_ID},{latest_return_Date}"
                write_db.write(entry)
                write_db.write("\n")
                return f"Game rented successfully! Latest return date: {latest_return_Date}"
            
        else: 
            return "Failed to rent! Game does not exist!"

    else: 
       return "Failed to rent! Customer is not subscribed!"
    


# Allows store manager to see what Video games are avaliable for rent

def avaliable_vid_games(): 

    r = open("Rental.txt", "r")

    unavaliable = []
    avaliable = []

    for line in r.readlines()[1:]:
        if line.split(",")[2] == "N/A":
            unavaliable.append(line)

    r.close()

    r = open("Video_Game_Info.txt", "r")
    avaliable.append('GameID, Name, Platform, Genre')

    for line in r.readlines()[1:]:
        line.replace('\n', '')
        line = line.split(",")
        line.pop(4)
        line = ', '.join(line)
        avaliable.append(line)

    for entry in unavaliable:
        for line in avaliable:
            if entry.split(",")[0] == line.split(", ")[0]:
                avaliable.remove(line)
    
    return avaliable



# Allows store manager to see what Board games are avaliable for rent 

def avaliable_brd_games(): 

    r = open("Rental.txt", "r")

    unavaliable = []
    avaliable = []

    for line in r.readlines()[1:]:
        if line.split(",")[2] == "N/A":
            unavaliable.append(line)

    r.close()

    r = open("Board_Game_Info.txt", "r")
    avaliable.append("GameID, Name, No.Players, Genre")

    for line in r.readlines()[1:]:
        line.replace('\n', '')
        line = line.split(",")
        line.pop(4)
        line = ', '.join(line)
        avaliable.append(line)

    for entry in unavaliable:
        for line in avaliable:
            if entry.split(",")[0] == line.split(", ")[0]:
                avaliable.remove(line)
      
    return avaliable


In [ ]:
import subscriptionManager as smSL
import feedbackManager as fmSL
import datetime 

from datetime import datetime



# Allows a user who rented a game to return it 


def game_Return(game_ID, user_ID):

    # Creating a list containing Rental.txt entries: 

    r = open("Rental.txt", "r")
    file_lst = r.readlines()[1:]
    r.close()

    # Fetching and formatting current date for return date:

    current_date = datetime.now().strftime("%Y-%m-%d")

    # Creating a list containing Return history for the specified game:

    game_return_history = []
    for line in file_lst:
        if line.split(",")[0] == game_ID:
            game_return_history.append(line.split(",")[2])

    # Checking to see if the game was out for return:
    
    if "N/A" in game_return_history:

        # Rewriting Rental.txt to fill return date entry:

        w = open("Rental.txt", "w")

        w.write('GameID,RentalDate,ReturnDate,RentalCustomerID,LatestReturnDate\n')

        for line in file_lst: 

            game_id = line.split(",")[0]
            rental_date = line.split(",")[1]
            return_date = line.split(",")[2]
            user_id = line.split(",")[3]
            latest_date = line.split(",")[4]

            if all([game_id == game_ID, user_id == user_ID, return_date == "N/A"]):

                w.write(f"{game_id},{rental_date},{current_date},{user_id},{latest_date}")

                concat_latest_date = int(latest_date.replace("-", ""))
                concat_return_date = int(current_date.replace("-", ""))

                # Once returned, checking to see if the game was returned late:
                
                if concat_return_date > concat_latest_date: 
                    return f"Customer has returned the game late!\nReturn Date: {current_date}\nLatest Return Date: {latest_date}"
                    
                else: 
                    return f"Game successfully returned!\nRental Date: {rental_date}\nReturn Date: {current_date}"
                
            else: 
                w.write(line)
    
        w.close()

        # Adding rating and comment to Game_Feedback.txt upon return:

        # Use radio buttons or something in the GUI to ensure only 1-5 can be inputted
        rating = int(input("Please enter a star rating 1-5: "))
        comments = str(input("Please enter a short comment on the game: "))

        w = open("Game_Feedback.txt", "a")
        w.write(f'\n{game_ID},{rating},{comments}')
        w.close()

        return "Rating and comment submitted!"

    else: 
        return "Game is not out for return!"


In [ ]:

import ipywidgets as widgets 
from ipywidgets import Layout 
from ipywidgets import Box
from ipywidgets import VBox



# Below we are defining all the widgets that appear in the GUI (There will be more to come)

rent_button = widgets.Button(
    value=False,
    description='Rent Games',
    disabled=False,
)

return_button = widgets.Button(
    value=False,
    description='Return Games',
    disabled=False
)

sessions_button = widgets.Button(
    value=False,
    description='Book Sessions',
    disabled=False,
)

pruning_button = widgets.Button(
    value=False,
    description='Prune Inventory',
    disabled=False
)

exit_button = widgets.Button(
    value=False,
    description='Exit',
    disabled=False
)

game_ID_text = widgets.Text(
    placeholder="Enter Game ID",
    disabled=False
)

user_ID_text = widgets.Text(
    placeholder="Enter User ID",
    disabled=False
)

game_search_submit_button = widgets.Button(
    value=False,
    description='Submit',
    disabled=False
)

view_games_button = widgets.Button(
    value=False, 
    description='Avaliable Games',
    disabled=False
)

board_games_button = widgets.Button(
    value=False, 
    description='Board Games',
    disabled=False
)

video_games_button = widgets.Button(
    value=False, 
    description='Video Games',
    disabled=False
)

game_return_submit_button = widgets.Button(
    value=False,
    description='Submit',
    disabled=False
)



# Below we are the output widgets for each of the widgets:

rent_output = widgets.Output()
return_output = widgets.Output()
pruning_output = widgets.Output()
sessions_output = widgets.Output()
avaliable_games_output = widgets.Output()
board_games_output = widgets.Output()



# Box containing the 5 initial buttons 

button_box = Box(children=[view_games_button, rent_button, return_button, sessions_button, pruning_button],
                 layout=Layout(display='flex',
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width = '200px'))

# Box containing the output widgets for each initial button

output_box = Box(children=[avaliable_games_output, rent_output, return_output, sessions_output, pruning_output],
                 layout=Layout(display='flex',
                               flex_flow='column wrap',
                               align_items='center',
                               align_content='center'))

# Box containing the output widgets for the initial Rent button

rent_box = Box(children=[game_ID_text, user_ID_text, game_search_submit_button, exit_button],
               layout=Layout(display='flex',
                             flex_flow='row wrap',
                             align_items='center',
                             align_content='center',
                             width='200px'))

# Box containing the output widgets for the initial Return button 

return_box = Box(children=[game_ID_text, user_ID_text, game_return_submit_button, exit_button],
                 layout=Layout(display='flex',
                               flex_flow='row wrap',
                               justify_content='center',
                               align_items='center',
                               align_content='center',
                               width='200px'))

# Box containing the output widgets for the initial View Games Button 

view_games_box = Box(children=[board_games_button, video_games_button, exit_button],
                     layout=Layout(display='flex',
                                   flex_flow='row wrap',
                                   justify_content='center',
                                   align_items='center',
                                   align_content='center',
                                   width='200px'))

# Box which contains the entire GUI 

universe_box = Box(children=[button_box, output_box],
                   layout=Layout(display='flex',
                                 flex_flow='row wrap',
                                justify_content='center',
                                align_items='center',
                                align_content='center',
                                width='100%',
                                height='100%'))




# Below are the functions defining the functionality of the widgets 


def on_rent_button_clicked(b):

    rent_box.layout.display = 'flex'
    button_box.layout.display = 'none'
    rent_output.layout.display = 'flex'
    rent_output.layout.justify_content='center'
    rent_output.layout.align_items='center'
    rent_output.layout.align_content='center'
    with rent_output:
        display(rent_box)


rent_button.on_click(on_rent_button_clicked)


def on_return_button_clicked(b):

    return_box.layout.display = 'flex'
    button_box.layout.display = 'none'
    return_output.layout.display = 'flex'
    return_output.layout.justify_content='center'
    return_output.layout.align_items='center'
    return_output.layout.align_content='center'
    with return_output:
        display(return_box)


return_button.on_click(on_return_button_clicked)


def on_pruning_button_clicked(b):

    button_box.layout.display = 'none'
    pruning_output.layout.display = 'flex'
    with pruning_output:
        print("This is pruning output")
        display(exit_button)


pruning_button.on_click(on_pruning_button_clicked)


def on_sessions_button_clicked(b):

    button_box.layout.display = 'none'
    sessions_output.layout.display = 'flex'
    with sessions_output:
        print("This is sessions output")
        display(exit_button)


sessions_button.on_click(on_sessions_button_clicked)


def on_avaliable_games_clicked(b):

    video_games_button.layout.display = 'block'
    board_games_button.layout.display = 'block'
    button_box.layout.display = 'none'
    avaliable_games_output.layout.display = 'flex'
    with avaliable_games_output: 
        display(view_games_box) 


view_games_button.on_click(on_avaliable_games_clicked)


def on_board_games_button_clicked(b):

    video_games_button.layout.display = 'none'
    board_games_button.layout.display = 'none'
    with avaliable_games_output: 
        avaliable_board_games = VBox(children=[widgets.Label(entry) for entry in avaliable_brd_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='250px'))
       
        display(avaliable_board_games)


board_games_button.on_click(on_board_games_button_clicked)


def on_video_games_button_clicked(b):

    video_games_button.layout.display = 'none'
    board_games_button.layout.display = 'none'
    with avaliable_games_output: 
        avaliable_video_games = VBox(children=[widgets.Label(entry) for entry in avaliable_vid_games()],
                                    layout=Layout(display='flex',
                                    flex_flow='row wrap',
                                    justify_content='flex-start',
                                    align_items='center',
                                    align_content='center',
                                    width='300px'))
        
        display(avaliable_video_games)


video_games_button.on_click(on_video_games_button_clicked)


def on_game_search_submit_button_clicked(b):

    rent_output.clear_output()
    with rent_output: 
        out = widgets.Text(value=game_Rent(user_ID_text.value, game_ID_text.value), disabled=True)
        display(out)
        display(exit_button)


game_search_submit_button.on_click(on_game_search_submit_button_clicked)


def on_game_return_submit_button_clicked(b):
    
    return_output.clear_output()
    with return_output:
        out = widgets.Text(value=game_Return(game_ID_text.value, user_ID_text.value), disabled=True)
        display(out)
        display(exit_button)


game_return_submit_button.on_click(on_game_return_submit_button_clicked)


def on_exit_button_clicked(b):

    sessions_output.clear_output()
    pruning_output.clear_output()
    return_output.clear_output()
    rent_output.clear_output()
    avaliable_games_output.clear_output()

    sessions_output.layout.display = 'none'
    pruning_output.layout.display = 'none'
    return_output.layout.display = 'none'
    rent_output.layout.display = 'none'
    avaliable_games_output.display = 'none'
    button_box.layout.display = 'flex'


exit_button.on_click(on_exit_button_clicked)


# Initial state of the GUI

display(universe_box)

